In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
links = fetch_website_links("https://www.parleproducts.com")
links



['/mrprevision',
 'JavaScript:Void(0);',
 '/index.aspx',
 'javascript:void(0);',
 '/index.aspx',
 'aboutus.aspx?mpgid=2&pgidtrail=2',
 '/aboutus.aspx?mpgid=2pgidtrail=2',
 '/aboutus.aspx?mpgid=2pgidtrail=2#legacy',
 '/csr.aspx?mpgid=7pgidtrail=7',
 '/csr.aspx?mpgid=7pgidtrail=7',
 '/worldwide-presence.aspx?mpgid=2pgidtrail=5',
 'brands.aspx?mpgid=3&pgidtrail=3',
 '/brands.aspx?mpgid=3&pgidtrail=3&catid=1',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=46',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=2',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=3',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=5',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=1',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=11',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=10',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=12',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=8',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=37',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=7',
 '/product.aspx?mpgid=3&pgidtrail=3&brandid=9',
 '/produc

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://www.parleproducts.com"))


Here is the list of links on the website https://www.parleproducts.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/mrprevision
JavaScript:Void(0);
/index.aspx
javascript:void(0);
/index.aspx
aboutus.aspx?mpgid=2&pgidtrail=2
/aboutus.aspx?mpgid=2pgidtrail=2
/aboutus.aspx?mpgid=2pgidtrail=2#legacy
/csr.aspx?mpgid=7pgidtrail=7
/csr.aspx?mpgid=7pgidtrail=7
/worldwide-presence.aspx?mpgid=2pgidtrail=5
brands.aspx?mpgid=3&pgidtrail=3
/brands.aspx?mpgid=3&pgidtrail=3&catid=1
/product.aspx?mpgid=3&pgidtrail=3&brandid=46
/product.aspx?mpgid=3&pgidtrail=3&brandid=2
/product.aspx?mpgid=3&pgidtrail=3&brandid=3
/product.aspx?mpgid=3&pgidtrail=3&brandid=5
/product.aspx?mpgid=3&pgidtrail=3&brandid=1
/product.aspx?mpgid=3&pgidtrail=3&brandid=11
/product.aspx?mpgid=3&pgidtrail=3&brandid=10
/product.aspx?mpgid=3&pgidt

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [9]:
select_relevant_links("https://www.parleproducts.com")

{'links': [{'type': 'about page',
   'url': 'https://www.parleproducts.com/about'},
  {'type': 'about page (legacy)',
   'url': 'https://www.parleproducts.com/about#legacy'},
  {'type': 'worldwide presence',
   'url': 'https://www.parleproducts.com/worldwide-presence.aspx?mpgid=2pgidtrail=5'},
  {'type': 'csr page',
   'url': 'https://www.parleproducts.com/csr.aspx?mpgid=7pgidtrail=7'},
  {'type': 'brands page',
   'url': 'https://www.parleproducts.com/brands.aspx?mpgid=3&pgidtrail=3'},
  {'type': 'brands - biscuits',
   'url': 'https://www.parleproducts.com/brands/biscuits'},
  {'type': 'brands - confectionery',
   'url': 'https://www.parleproducts.com/brands/confectionery'},
  {'type': 'brands - snacks',
   'url': 'https://www.parleproducts.com/brands/snacks'},
  {'type': 'brands - cakes',
   'url': 'https://www.parleproducts.com/brands/parle-cakes'},
  {'type': 'brands - rusk',
   'url': 'https://www.parleproducts.com/brands/rusk'},
  {'type': 'brands - platina range',
   'url': 'ht

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [11]:
select_relevant_links("https://www.parleproducts.com")

Selecting relevant links for https://www.parleproducts.com by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'about page',
   'url': 'https://www.parleproducts.com/about'},
  {'type': 'brands page',
   'url': 'https://www.parleproducts.com/brands.aspx?mpgid=3&pgidtrail=3'},
  {'type': 'csr page', 'url': 'https://www.parleproducts.com/csr'},
  {'type': 'global presence page',
   'url': 'https://www.parleproducts.com/global-presence'},
  {'type': 'worldwide presence page',
   'url': 'https://www.parleproducts.com/worldwide-presence.aspx?mpgid=2&pgidtrail=5'},
  {'type': 'career page',
   'url': 'https://www.parleproducts.com/career.aspx?mpgid=15&pgidtrail=15'},
  {'type': 'news page', 'url': 'https://www.parleproducts.com/news'},
  {'type': 'Parle G promo video (YouTube)',
   'url': 'https://www.youtube.com/watch?v=Mcv9owdYwx0'},
  {'type': 'Facebook page', 'url': 'https://www.facebook.com/ParleFamily/'},
  {'type': 'Twitter profile',
   'url': 'https://twitter.com/parlefamily?lang=en'},
  {'type': 'Instagram profile',
   'url': 'https://www.instagram.com/parleproducts/?hl=e

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://www.parleproducts.com"))

Selecting relevant links for https://www.parleproducts.com by calling gpt-5-nano
Found 24 relevant links
## Landing Page:


	Parle G Biscuit Company - Leading Manufacturer of Snacks


×
About Us
Company Overview
India's leading manufacturer of biscuits and confectionery.
Parle's Legacy
The Legacy of
90
years
Parle
Corporate Social Responsibility
Worldwide Presence
Parle biscuits and confectionaries are fast gaining acceptance in International markets...
Brands
Biscuits
Parle-G Gluco
Parle-G
20-20 Cookies
KrackJack
Monaco
Magix Creme
Fab! Range
Parle Marie
Hide & Seek Milano
Milk Shakti
Hide & Seek
Nutricrunch
Happy Happy
Parle Coconut Biscuits
Parle Top
Confectionery
Duet
New
2 in 1 Eclairs
Kaccha Mango Bite
Cafechino
Londonderry
Kismi
Orange Bite
Melody
Mango Bite
Poppins
Rol-a-Cola
Mazelo Fruit Gang
Mazelo
Kapi Candy
Fusion
Snacks
Mexitos
Fulltoss
Parle's Wafers
Chatkeens
Cakes
Happy Happy Cakes
Rusk
Parle Rusk
Platina Range
Hide & Seek
Hide & Seek Milano
Nutricrunch
Mexitos
Nutricru

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("parle", "https://www.parleproducts.com")

Selecting relevant links for https://www.parleproducts.com by calling gpt-5-nano
Found 15 relevant links


"\nYou are looking at a company called: parle\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\n\r\n\tParle G Biscuit Company - Leading Manufacturer of Snacks\r\n\n\n×\nAbout Us\nCompany Overview\nIndia's leading manufacturer of biscuits and confectionery.\nParle's Legacy\nThe Legacy of\n90\nyears\nParle\nCorporate Social Responsibility\nWorldwide Presence\nParle biscuits and confectionaries are fast gaining acceptance in International markets...\nBrands\nBiscuits\nParle-G Gluco\nParle-G\n20-20 Cookies\nKrackJack\nMonaco\nMagix Creme\nFab! Range\nParle Marie\nHide & Seek Milano\nMilk Shakti\nHide & Seek\nNutricrunch\nHappy Happy\nParle Coconut Biscuits\nParle Top\nConfectionery\nDuet\nNew\n2 in 1 Eclairs\nKaccha Mango Bite\nCafechino\nLondonderry\nKismi\nOrange Bite\nMelody\nMango Bite\nPoppins\nRol-a-Cola\nMazelo Fruit Gang\nMazelo\nKapi Candy\nFu

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("parle", "https://www.parleproducts.com")

Selecting relevant links for https://www.parleproducts.com by calling gpt-5-nano
Found 15 relevant links


# Parle G Biscuit Company  
*Leading Manufacturer of Snacks & Confectionery in India for 90 Years*

---

## About Parle  

Parle G Biscuit Company is India's foremost manufacturer of biscuits, confectionery, and snacks, with a rich legacy spanning 90 years. Renowned for its trusted brands and iconic products, Parle has played an integral role in the daily lives of millions across India and is rapidly expanding its presence internationally.  

With a product portfolio that caters to diverse tastes and preferences, Parle offers an extensive range of biscuits, confectionery, snacks, and breakfast cereals, making it one of the most loved and recognized brands in the FMCG sector.

---

## Legacy & Values  

- **90 Years of Excellence:** A heritage built on quality, trust, and innovation.  
- **Commitment to Quality:** Ensuring the highest standards in manufacturing and product development.  
- **Customer-first Approach:** Offering affordable and accessible food products to all segments of society.  
- **Corporate Social Responsibility:** Actively engaged in community development and sustainability initiatives.

---

## Global Reach  

Parle brands have gained significant acceptance beyond Indian borders, with an expanding international footprint. Its presence in overseas markets highlights its ambition to become a global leader in biscuits and confectionery.

---

## Product Portfolio  

### Biscuits  
- Parle-G Gluco  
- Parle-G  
- 20-20 Cookies  
- KrackJack  
- Monaco  
- Magix Creme  
- Fab! Range  
- Parle Marie  
- Hide & Seek Milano  
- Milk Shakti  
- Hide & Seek  
- Nutricrunch  
- Happy Happy  
- Parle Coconut Biscuits  
- Parle Top  

### Confectionery  
- Duet  
- 2 in 1 Eclairs  
- Kaccha Mango Bite  
- Cafechino  
- Londonderry  
- Kismi  
- Orange Bite  
- Melody  
- Mango Bite  
- Poppins  
- Rol-a-Cola  
- Mazelo Fruit Gang  
- Mazelo  
- Kapi Candy  
- Fusion  

### Snacks & Others  
- Mexitos  
- Fulltoss  
- Parle's Wafers  
- Chatkeens  
- Happy Happy Cakes  
- Parle Rusk  
- Hide & Seek Fills  
- Atta & Breakfast Cereals  
- Parle Gol Gappa Candy  
- Crunchee Munchee  
- Zing  
- Parle Suraksha  

---

## Company Culture & Careers  

Parle fosters a culture of innovation, integrity, and inclusiveness. The company is committed to nurturing its workforce through continuous learning and development opportunities, promoting teamwork, and encouraging employees to contribute ideas for growth and sustainability.  

Working at Parle means being part of a legacy brand that values dedication and provides avenues for personal and professional growth. Parle offers exciting career prospects in manufacturing, marketing, R&D, supply chain, and corporate functions.

---

## Corporate Social Responsibility  

Parle actively contributes to social welfare and sustainable development initiatives. The company focuses on areas such as education, health, hygiene, and environmental conservation to positively impact communities it serves.

---

## Why Choose Parle?  

- Trusted brand with a 90-year heritage  
- Wide range of high-quality, affordable products  
- Strong commitment to innovation and customer satisfaction  
- Expanding global footprint and growing international acceptance  
- Progressive workplace culture with career growth opportunities  
- Responsible corporate citizen through CSR activities  

---

For more information and to explore career opportunities, visit [Parle Official Website](https://www.parleproducts.com).  

---

*Parle G Biscuit Company — Bringing Joy and Nourishment Across Generations*

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("parle", "https://www.parleproducts.com")

Selecting relevant links for https://www.parleproducts.com by calling gpt-5-nano
Found 12 relevant links


# Parle G Biscuit Company Brochure

---

## About Parle

Founded over **90 years ago**, Parle is India's leading manufacturer of biscuits, confectionery, and snacks. With a rich heritage and a passion for quality, Parle has become a household name synonymous with trust, taste, and innovation.

---

## Our Legacy

- Over **90 years** of excellence in the food industry
- Pioneer in affordable, high-quality biscuits and snacks for all age groups
- A legacy rooted in bringing delightful flavors to millions of customers worldwide

---

## Product Portfolio

Parle offers a diverse range of products across several categories:

### Biscuits
- **Parle-G Gluco & Parle-G** (Iconic biscuits)
- 20-20 Cookies
- KrackJack, Monaco, Magix Creme, Fab! Range
- Parle Marie, Hide & Seek Milano, Milk Shakti, Nutricrunch
- Happy Happy, Parle Coconut Biscuits, Parle Top

### Confectionery
- Duet, 2 in 1 Eclairs, Kaccha Mango Bite
- Cafechino, Londonderry, Kismi, Orange Bite
- Melody, Mango Bite, Poppins, Rol-a-Cola
- Mazelo Fruit Gang, Mazelo, Kapi Candy, Fusion

### Snacks
- Mexitos, Fulltoss, Parle's Wafers, Chatkeens

### Cakes & Rusk
- Happy Happy Cakes, Parle Rusk, Platina Range

### Breakfast & Other Foods
- Atta flour, Breakfast Cereals, ASAP products

---

## Global Presence

Parle's products are increasingly gaining acceptance in **international markets**, reflecting the company's commitment to quality and taste. Parle continues its expansion globally, delighting consumers beyond India.

---

## Corporate Social Responsibility (CSR)

Parle is committed to giving back to society through various CSR initiatives. While specifics are not detailed here, Parle integrates responsible practices in manufacturing and community engagement, supporting sustainable development alongside business growth.

---

## Company Culture & Careers

- Parle emphasizes a strong legacy of **innovation, quality, and customer-centricity.**
- The company encourages talent to join a dynamic team dedicated to creating beloved products.
- Career opportunities exist across multiple functions - manufacturing, R&D, marketing, sales, and corporate roles.
- Prospective candidates can look forward to contributing to a brand with deep roots and expanding global ambitions.

---

## Why Choose Parle?

- Trusted brand with **nearly a century of expertise**
- Wide range of innovative and wholesome products catering to all tastes and age groups
- Strong presence in both domestic and international markets
- Commitment to quality, affordability, and social responsibility

---

## Contact & More Information

For more details about our products, careers, or corporate initiatives:

- Visit us at [Parle G's website](https://www.parle.com)  
- Explore our product recipes, news, and media resources  
- Join us in our journey of bringing joy and nutrition to families worldwide

---

**Parle G Biscuit Company**  
*Leading India’s favourite snacks for over 90 years*